In [1]:
import pandas as pd
from pathlib import Path

In [9]:
# res_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/comparison_inc/test_etpr/2025-08-15T13:40:51-Qwen2.5-3B-Instruct/test_etpr_output_extracted_answers_last.parquet')
res_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/comparison_inc/test_etpr/2025-09-18T17:42:36-NACC_Inc-sce-tanh-1000/test_etpr_output_extracted_answers_last.parquet')

df = pd.read_parquet(res_path)

df['correctness'] = (df['ground_truth'] == df['prediction']).astype(int) 

In [10]:
df_minimal = df[['ID','ground_truth_text','correctness']]
df_minimal

,ID,ground_truth_text,correctness
0,NACC577827,Frontotemporal lobar degeneration and its vari...,0
1,NACC577827,Frontotemporal lobar degeneration and its vari...,0
2,NACC577827,Frontotemporal lobar degeneration and its vari...,0
3,NACC577827,Frontotemporal lobar degeneration and its vari...,0
4,NACC577827,Frontotemporal lobar degeneration and its vari...,0
...,...,...,...
28125,NACC072037,Not applicable (no cognitive impairment),1
28126,NACC072037,Not applicable (no cognitive impairment),1
28127,NACC072037,Not applicable (no cognitive impairment),1
28128,NACC072037,Not applicable (no cognitive impairment),1


In [15]:
np.arange(3,3)

array([], dtype=int64)

In [18]:
def f(c,n=5,k=1):
    return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))

In [26]:
f(5)

np.float64(1.0)

In [33]:
import numpy as np

def pass_at_k(df, n, k=1): 
    """ 
    :param n: total number of samples 
    :param c: number of correct samples 
    :param k: k in pass@$k$ """
    # print(n)
    # cs = df.groupby('problem').sum('correctness')
    cs = df.groupby('ID').sum('correctness') # problem and ID are the same thing
    vals = []
    for i, row in cs.iterrows():
        c = row['correctness']
        # print(n,c,k)
        if n - c < k: vals.append(1.0)
        else: vals.append(1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1)))
         
    # print(vals)
    return np.mean(vals)


def combine_results_by_ground_truth(df, k):
    """
    Like combine_results, but computes pass@1 and cons@k separately for each ground_truth.
    Returns a DataFrame with columns: ['ground_truth', 'metric', 'model', 'score']
    """
    # Iterate over each model and its corresponding DataFrame in the results dictionary
    # For each unique ground truth label in the DataFrame

    all_records = []

    for gt in df['ground_truth_text'].unique():
        # Filter the DataFrame for the current ground truth
        df_gt = df[df['ground_truth_text'] == gt].reset_index(drop=True)
        
        # Assign a number to each ID. Why ado this if ID is already unique?
        # df_gt['problem'] = df_gt.groupby('ID').ngroup() 

        # Number of unique problems
        n = len(df_gt['ID'].unique())
        # Number of attempts per problem (assumes equal number for all)
        p = len(df_gt) // n
        # print(gt, n, p)  # For debugging: show current ground truth, n, and p

        # Compute pass@k score for this ground truth and model
        pass_score = pass_at_k(df_gt, p, k)

        # Append pass@1 result to the records list
        all_records.append({
            'ground_truth': gt,
            'metric': 'pass@1',
            # 'model': model_name,
            'score': pass_score,
            'size': len(df_gt),
        })

    return all_records


In [34]:
metrics = combine_results_by_ground_truth(df_minimal,k=1)

In [35]:
pd.DataFrame(metrics)

,ground_truth,metric,score,size
0,Frontotemporal lobar degeneration and its vari...,pass@1,0.293808,3795
1,Alzheimer's disease (AD),pass@1,0.657223,16095
2,Not applicable (no cognitive impairment),pass@1,0.717140,4755
3,Lewy body disease (LBD),pass@1,0.733659,2050
4,"Other (Multiple system atrophy, Essential trem...",pass@1,0.032258,465
5,Vascular brain injury or vascular dementia inc...,pass@1,0.693243,740
6,Psychiatric conditions including schizophrenia...,pass@1,0.016667,60
7,Systemic and environmental factors including i...,pass@1,0.011765,170


In [6]:
# df_long = combine_results_by_ground_truth(modified_models, k=1)

In [7]:
# sahana is doing this, which is macro average
# df_long.groupby('model')['score'].mean().sort_values().to_csv(f"../llm_answer_extractor/outputs_sub/subgroups/{benchmark_type}/{eval_type.lower()}.csv")